In [1]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from sklearn.metrics import accuracy_score

d:\GitHub\FedGNN-CKD\fedgnn_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
class GCN(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, output_dim)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, training=self.training)
        x = self.conv2(x, edge_index)
        return x

In [5]:
# Load graph (Hospital A for testing)
graph = torch.load("../data/graph/graph_A.pt", weights_only=False).to(device)
input_dim = graph.num_node_features

# Load global model
global_model = GCN(input_dim, 32, 2).to(device)
global_model.load_state_dict(
    torch.load("../data/models/global_fedavg_model.pth")
)

global_model.eval()
with torch.no_grad():
    logits = global_model(graph.x, graph.edge_index)
    preds = logits.argmax(dim=1)
    acc = accuracy_score(graph.y.cpu(), preds.cpu())

print("Global Model Accuracy:", acc)

Global Model Accuracy: 0.98125
